**Block 1: Import Pandas + the small classes**

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


class Roommate:
    def __init__(self, name):
        self.name = name
        self.balance = 0.0

    def __str__(self):
        return self.name + " (balance: $" + format(self.balance, ".2f") + ")"


class Expense:
    def __init__(self, item, cost, payer, shares, split_type):
        self.item = item
        self.cost = cost
        self.payer = payer
        self.shares = shares
        self.split_type = split_type
        self.paid_back = set()

    def unpaid(self):
        names = []
        for name in self.shares:
            if name != self.payer and name not in self.paid_back:
                names.append(name)
        return names


class Chore:
    def __init__(self, name, points):
        self.name = name
        self.points = points
        self.active = True


class SharedItem:
    def __init__(self, name):
        self.name = name
        self.status = "In stock"
        self.last_bought_by = "-"

**Block 2: The Household class**

In [ ]:
class Household:
    def __init__(self):
        self.roommates = []
        self.purchases = []
        self.chores = [Chore("Take out trash", 1),
                       Chore("Wipe kitchen counters", 1),
                       Chore("Vacuum living room", 2),
                       Chore("Do the dishes", 2),
                       Chore("Clean bathroom", 3)]
        self.supplies = [SharedItem("Paper towels"),
                         SharedItem("Dish soap"),
                         SharedItem("Trash bags")]

    def get_names(self):
        names = []
        for r in self.roommates:
            names.append(r.name)
        return names

    def add_roommate(self, name):
        name = name.strip()
        if name == "":
            print("Name can't be blank.")
            return False
        if name in self.get_names():
            print(f"{name} is already in the household.")
            return False
        self.roommates.append(Roommate(name))
        return True

    def find_roommate(self, name):
        for r in self.roommates:
            if r.name == name:
                return r
        return None

    def add_purchase(self, item, cost, payer_name, shares, split_type):
        payer = self.find_roommate(payer_name)
        payer.balance += cost
        for name in shares:
            self.find_roommate(name).balance -= shares[name]

        self.purchases.append(Expense(item, cost, payer_name, shares, split_type))
        print(f"\n{item}: ${cost:.2f} paid by {payer_name} ({split_type} split)")
        for name in shares:
            print(f"  {name}'s share: ${shares[name]:.2f}")

    def purchases_table(self):
        rows = []
        for p in self.purchases:
            still_owe = p.unpaid()
            if len(still_owe) == 0:
                status = "All paid back"
            else:
                status = "Waiting on " + ", ".join(still_owe)
            rows.append({"Item": p.item, "Cost": p.cost, "Paid by": p.payer,
                         "Split": p.split_type, "Shared by": ", ".join(p.shares),
                         "Repayment": status})
        table = pd.DataFrame(rows)
        table.index = table.index + 1
        return table

    def balances_table(self):
        rows = []
        for r in self.roommates:
            if r.balance > 0.005:
                note = "is owed money"
            elif r.balance < -0.005:
                note = "owes money"
            else:
                note = "all even"
            rows.append({"Roommate": r.name, "Balance": round(r.balance, 2), "Status": note})
        table = pd.DataFrame(rows)
        table.index = table.index + 1
        return table

    def unpaid_debts(self):
        debts = []
        for p in self.purchases:
            for name in p.unpaid():
                debts.append((p, name, p.shares[name]))
        return debts

    def show_debts(self):
        debts = self.unpaid_debts()
        for i in range(len(debts)):
            p, name, amount = debts[i]
            print(f"{i + 1}. {name} owes {p.payer} ${amount:.2f} for {p.item}")
        return debts

    def record_repayment(self, debt):
        p, name, amount = debt
        p.paid_back.add(name)
        self.find_roommate(name).balance += amount
        self.find_roommate(p.payer).balance -= amount
        print(f"Marked as paid: {name} paid {p.payer} back ${amount:.2f} for {p.item}.")

    def reminders(self):
        debts = self.unpaid_debts()
        if len(debts) == 0:
            print("Everyone's paid back. No reminders needed!")
        for p, name, amount in debts:
            print(f"Reminder: Hey {name}! You still owe {p.payer} ${amount:.2f} for {p.item}.")

    def show_chores(self):
        print("\nChores:")
        for i in range(len(self.chores)):
            c = self.chores[i]
            if c.active:
                state = "ON"
            else:
                state = "off"
            print(f"{i + 1}. {c.name} ({c.points} pts) - {state}")

    def toggle_chore(self, number):
        c = self.chores[number - 1]
        c.active = not c.active
        if c.active:
            print(f"{c.name} is now ON.")
        else:
            print(f"{c.name} is now off.")

    def add_chore(self, name, points):
        name = name.strip()
        if name == "":
            print("Chore name can't be blank.")
            return False
        for c in self.chores:
            if c.name.lower() == name.lower():
                print(f"{name} is already on the chore list.")
                return False
        self.chores.append(Chore(name, points))
        print(f"Added: {name} ({points} pts)")
        return True

    def split_chores(self):
        totals = {}
        for r in self.roommates:
            totals[r.name] = 0

        rows = []
        for points in [3, 2, 1]:
            for c in self.chores:
                if c.active and c.points == points:
                    lowest = self.roommates[0].name
                    for name in totals:
                        if totals[name] < totals[lowest]:
                            lowest = name
                    totals[lowest] += c.points
                    rows.append({"Chore": c.name, "Points": c.points, "Assigned to": lowest})

        print("\nChore points per roommate:", totals)
        print("Note: a perfectly equal split isn't always possible.")
        table = pd.DataFrame(rows)
        table.index = table.index + 1
        return table

    def supplies_table(self):
        rows = []
        for s in self.supplies:
            rows.append({"Item": s.name, "Status": s.status, "Last bought by": s.last_bought_by})
        table = pd.DataFrame(rows)
        table.index = table.index + 1
        return table

    def restock_list(self):
        need = []
        for s in self.supplies:
            if s.status == "Low" or s.status == "Out":
                need.append(s.name)
        return need

**Block 3: Helper functions for safe input**

In [ ]:
def ask_number(prompt, minimum):
    while True:
        try:
            value = float(input(prompt))
            if value >= minimum:
                return value
            print(f"Please enter a number of at least {minimum}.")
        except ValueError:
            print("Please type a number, like 3 or 12.50.")


def ask_roommate(home, prompt):
    name = input(prompt)
    while name not in home.get_names():
        print("That name isn't in the household. Try again.")
        name = input(prompt)
    return name


def ask_choice(prompt, low, high):
    while True:
        try:
            choice = int(input(prompt))
            if low <= choice <= high:
                return choice
            print(f"Please pick a number from {low} to {high}.")
        except ValueError:
            print("Please type a whole number.")


def ask_yes_no(prompt):
    answer = input(prompt).strip().lower()
    while answer != "y" and answer != "n":
        print("Please type y or n.")
        answer = input(prompt).strip().lower()
    return answer == "y"


def ask_sharers(home):
    if ask_yes_no("Split among everyone? (y/n): "):
        return home.get_names()
    while True:
        text = input("Who should be left out? Type names separated by commas: ")
        left_out = []
        for name in text.split(","):
            left_out.append(name.strip())

        problem = False
        for name in left_out:
            if name not in home.get_names():
                print(f"{name} isn't in the household.")
                problem = True

        sharers = []
        for name in home.get_names():
            if name not in left_out:
                sharers.append(name)

        if len(sharers) == 0:
            print("At least one person has to share the cost.")
            problem = True
        if not problem:
            return sharers


def ask_shares(cost, sharers):
    shares = {}
    if ask_yes_no("Split evenly? (y/n): "):
        for name in sharers:
            shares[name] = cost / len(sharers)
        return shares, "Even"

    while True:
        print("Enter each person's percent. They must add up to 100.")
        total = 0
        for name in sharers:
            percent = ask_number(f"  {name}'s percent: ", 0)
            shares[name] = cost * percent / 100
            total += percent
        if abs(total - 100) < 0.01:
            return shares, "Custom"
        print(f"Those add up to {total}, not 100. Let's try again.")

**Block 4: Set up the household**

In [ ]:
home = Household()

num_roommates = int(ask_number("How many roommates live in the household? ", 1))

while len(home.roommates) < num_roommates:
    name = input(f"Enter the name of roommate {len(home.roommates) + 1}: ")
    home.add_roommate(name)

print("Roommates:", home.get_names())

**Block 5: The menu**

In [ ]:
while True:
    print("\n--- RoomieShare Menu ---")
    print("1. Add a shared purchase")
    print("2. Record a repayment (paid back)")
    print("3. Payment reminders")
    print("4. See purchase history")
    print("5. See balances")
    print("6. Turn a chore on/off")
    print("7. Add a custom chore")
    print("8. Split this week's chores")
    print("9. Update a supply")
    print("10. See supplies + restock list")
    print("11. Quit")
    choice = ask_choice("Choose 1-11: ", 1, 11)

    if choice == 1:
        item = input("What was purchased? ")
        cost = ask_number("How much did it cost? $", 0.01)
        payer = ask_roommate(home, "Who paid? ")
        sharers = ask_sharers(home)
        shares, split_type = ask_shares(cost, sharers)
        home.add_purchase(item, cost, payer, shares, split_type)

    elif choice == 2:
        if len(home.unpaid_debts()) == 0:
            print("Nobody owes anything right now.")
        else:
            debts = home.show_debts()
            number = ask_choice("Which one was paid back? ", 1, len(debts))
            home.record_repayment(debts[number - 1])

    elif choice == 3:
        home.reminders()

    elif choice == 4:
        if len(home.purchases) == 0:
            print("No purchases yet.")
        else:
            print(home.purchases_table())

    elif choice == 5:
        print(home.balances_table())

    elif choice == 6:
        home.show_chores()
        number = ask_choice("Which chore number? ", 1, len(home.chores))
        home.toggle_chore(number)

    elif choice == 7:
        name = input("New chore name: ")
        points = ask_choice("Effort points (1 = light, 2 = medium, 3 = heavy): ", 1, 3)
        home.add_chore(name, points)

    elif choice == 8:
        print(home.split_chores())

    elif choice == 9:
        print(home.supplies_table())
        number = ask_choice("Which supply number? ", 1, len(home.supplies))
        item = home.supplies[number - 1]
        status = ask_choice("New status: 1 = In stock, 2 = Low, 3 = Out: ", 1, 3)
        if status == 1:
            item.status = "In stock"
            item.last_bought_by = ask_roommate(home, "Who bought it? ")
        elif status == 2:
            item.status = "Low"
        else:
            item.status = "Out"
        print(f"{item.name} is now: {item.status}")

    elif choice == 10:
        print(home.supplies_table())
        need = home.restock_list()
        if len(need) == 0:
            print("Restock list: nothing needed!")
        else:
            print("Restock list:", need)

    else:
        print("Bye! See you next time.")
        break